In [6]:
# IMPORTS 
import numpy as np; import matplotlib.pyplot as plt; import matplotlib.colors as mcolors; import matplotlib.animation as animation; import warnings
from matplotlib.patches import Patch
from matplotlib.colors import LinearSegmentedColormap
from scipy.ndimage import gaussian_filter
from scipy.signal import convolve2d
warnings.filterwarnings('ignore')

np.random.seed(42)

plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor': '#0f1117',
    'axes.edgecolor': '#444',
    'axes.labelcolor': 'white',
    'text.color': 'white',
    'xtick.color': 'white',
    'ytick.color': 'white',
    'grid.color': '#333',
    'figure.dpi': 120,
    'font.family': 'monospace'
})

# The Mathematics of Avalanches
## A Physics-Based Simulation of Snow Slab Dynamics

---

> *"The mountain doesn't care about your skill level."* — Common freeskiing saying

---

Freeskiing is one of the few sports where the terrain itself is the arena, and also the threat. Every skier who ventures into steep, uncontrolled terrain, implicitly makes a risk calculation: *Is this slope stable? Will it hold?* Most of the time, this is intuitive. But intuition has limits, and avalanches don't.

Todorka Peak (2,746 m) in the Pirin mountain range, Bulgaria, is a classic example of high-consequence terrain. Its north-facing couloir (facing Bansko) holds deep snow throughout the winter, and its steep upper faces (30°–50°) place it squarely in the most dangerous avalanche angle range. Real incidents have occurred here.

This project asks: **can we mathematically model the conditions that cause a slope to fail, and simulate what happens when it does?**

We will:
1. Build a physically realistic terrain model of Todorka's topography
2. Derive and apply the physics of snow slab stability
3. Identify danger zones across the terrain grid
4. Simulate avalanche propagation using a cellular automata model
5. Interpret the results in practical terms

**This is not a toy model.** The physics here — slope mechanics, fracture thresholds, runout estimation — are the same foundations used by professional avalanche forecasters, used for evaluating snowpack and snowmass conditions of venues, hosting freeride events.

---
## Section 1 — Problem Formulation: When Does a Mountain 'Break'?

### 1.1 The Avalanche Problem

An avalanche is basically a **mechanical failure**. A mass of snow sits on a slope. Gravity pulls it downwards. Internal friction and cohesion holds it in place. When the pulling force exceeds the holding force, the snowpack fails, and you get in big trouble if you are sitting in the middle of it without the needed equipment.

Question is:

> **Given a terrain and snowpack properties, where is the slope on the verge of failure, and how far will the snow travel if it releases?**

This is both a **statics problem** (when does it fail?) and a **dynamics problem** (how does it propagate?).

### 1.2 Cellular Automata

The Cellular Automation avalanche modelling discretizes the terrain into a grid of cells. Each cell has a state (snow mass, elevation). At each time step, a simple set of local rules determines how mass moves between neighboring cells. Despite the simplicity of the rules, complex, realistic-looking global behavior emerges.

This is not the only  approach, but a well-established one in geomorphology and hazard simulation. The tradeoffs:

| Approach | Pros | Cons |
|---|---|---|
| Cellular Automata | Intuitive, fast, extensible | Approximate physics |
| Shallow Water Equations | Full fluid dynamics | Requires PDE solvers, much more complex |
| Discrete Element Methods | Most realistic | Computationally prohibitive |

### 1.3 Assumptions & Constraints

Our model makes the following explicit assumptions:
- Snow is treated as a granular, incompressible mass (no phase changes)
- Terrain is static (the avalanche doesn't erode the bed)
- Snowpack is uniform in density and depth (homogeneous slab assumption)
- We model a **dry slab avalanche** (the most common and deadly type, where a more dense layer sits on top of a weak layer)
- Wind and temperature effects are represented implicitly through the snow density parameter

---
## Section 2 — The Physics & Mathematics of Snow Slab Stability

### 2.1 Forces on an Inclined Plane

Consider a snow slab of thickness $h$ (meters), density $\rho$ (kg/m³), resting on a slope of angle $\theta$ (degrees).

The weight per unit area of the slab is:
$$W = \rho g h$$

This weight resolves into two components:

**Normal stress** (perpendicular to slope — compresses the snowpack into the slope):
$$\sigma = \rho g h \cos\theta$$

**Shear stress** (parallel to slope — the force trying to slide the slab downhill):
$$\tau = \rho g h \sin\theta$$

### 2.2 The Mohr-Coulomb Failure Criterion

Snow obeys the **Mohr-Coulomb failure criterion**. The maximum shear stress the snowpack can resist before failure is:

$$\tau_{\text{max}} = c + \sigma \tan\phi$$

Where:
- $c$ = **cohesion** (Pa) — snow's intrinsic strength, independent of normal load. Fresh dry snow: ~100–500 Pa. Settled snow: ~500–2000 Pa.
- $\phi$ = **internal friction angle** — typically 20°–35° for snow
- $\sigma$ = normal stress (from above)

### 2.3 The Factor of Safety

The **Factor of Safety (FoS)** is the central metric in slope stability analysis:

$$\text{FoS} = \frac{\tau_{\text{max}}}{\tau} = \frac{c + \rho g h \cos\theta \cdot \tan\phi}{\rho g h \sin\theta}$$

Interpretation:
- $\text{FoS} > 1$ → **Stable** (safety margin)
- $\text{FoS} \sim 1$ → **Marginal** (sensitive to disturbance)
- $\text{FoS} < 1$ → **Unstable** (failure imminent)

**Note:** A skier crossing the slope is essentially a dynamic load that locally reduces FoS — this is why a single person can trigger a slope that appeared stable.

### 2.4 Critical Angle Derivation

For a purely cohesionless snowpack ($c = 0$), the stability condition simplifies elegantly:

$$\text{FoS} = \frac{\tan\phi}{\tan\theta}$$

Failure occurs when $\text{FoS} \leq 1$, giving the **critical angle**:

$$\theta_c = \phi$$

For snow with $\phi \approx 30°$, any slope steeper than 30° is potentially unstable if cohesion is low enough — consistent with the well-known empirical rule that most avalanches release on 30°–45° slopes.

### 2.5 The α-Angle Runout Model

Once an avalanche releases, how far does it travel? The **α-angle method** (Runout Ratio Model) is the simplest physically grounded estimate:

$$\tan\alpha = \frac{H}{L}$$

Where $H$ is the total vertical drop and $L$ is the horizontal runout distance. Empirically, $\alpha \approx 18°$–22° for most avalanche paths. This gives us a baseline expectation for our simulation to validate against.

---
## Section 3 — Terrain Model: Todorka Peak, Pirin

### 3.1 Terrain Generation

Since we can't pull live SRTM data in this environment, we construct a **physically faithful synthetic DEM** (Digital Elevation Model) for Todorka Peak.

The terrain is built from:
- A base mountain shape set at Todorka's real elevation (2,746 m)
- Ridge lines reflecting the real NW and NE ridges

In [10]:
# TERRAIN GENERATION — Todorka Peak synthetic DEM

GRID_SIZE = 200        # 200 x 200 cells
CELL_SIZE = 15.0       # meters per cell → 3km x 3km domain
SUMMIT_ELEV = 2746.0   # Todorka summit, meters
BASE_ELEV = 1950.0     # approximate treeline / valley floor


x = np.linspace(-1, 1, GRID_SIZE)
y = np.linspace(-1, 1, GRID_SIZE)
X, Y = np.meshgrid(x, y)

# base mountain
r_ellipse = np.sqrt((X / 0.55)**2 + (Y / 0.85)**2)
base_mountain = np.exp(-2.2 * r_ellipse**2)

# NW ridge
nw_ridge = 0.18 * np.exp(-((X + 0.25 + Y * 0.6)**2) / 0.03)

#  NE Shoulder 
ne_shoulder = 0.12 * np.exp(-((X - 0.3 - Y * 0.4)**2) / 0.025)

# S-facing bowl 
s_bowl = -0.06 * np.exp(-((Y + 0.5)**2 + X**2) / 0.08)


def make_noise(scale, amplitude):
    raw = np.random.randn(GRID_SIZE, GRID_SIZE)
    return amplitude * gaussian_filter(raw, sigma=scale)

noise = (make_noise(12, 0.07) +    # 
         make_noise(4,  0.025) +   # 
         make_noise(1.5, 0.008))   #

#  comp combination 
normalized_elev = base_mountain + nw_ridge + ne_shoulder + s_bowl + noise
normalized_elev = np.clip(normalized_elev, 0, None)

# Scale to real elevation
elev_range = SUMMIT_ELEV - BASE_ELEV
DEM = BASE_ELEV + (normalized_elev / normalized_elev.max()) * elev_range
DEM = gaussian_filter(DEM, sigma=1.2)  

print(f"DEM shape: {DEM.shape}")
print(f"Elevation range: {DEM.min():.0f} m — {DEM.max():.0f} m")
print(f"Grid domain:     {GRID_SIZE * CELL_SIZE / 1000:.1f} km × {GRID_SIZE * CELL_SIZE / 1000:.1f} km")